# Tumor-Stroma Mask Segmentation with Pre-Trained U-Net

**Author: Fabien Bertillot**  

---

## 1. Purpose

Generate high-resolution binary masks (tumor vs stroma) for TMA cores using a pretrained U-Net.
The notebook is **inference-only**: no weight updates occur.

---

## 2. Requirements

- **Python 3.9+**  
- Packages:  
  - `tensorflow` / `keras` (U-Net)  
  - `numpy`, `pandas`, `tifffile`, `scikit-image`, `opencv-python`  
- **Pretrained weights:** `.h5`  
- **Input directory** of images (`.tif`)  

---

## 3. Workflow Overview

1. **Load dependencies**  
2. **Configure inputs and load the U-Net model**  
3. **Run predictions**  
   - Label mask per image (`<image name>.png`) written to `MASK_DIR`  
---


## 1) Load Dependencies

In [ ]:
from pathlib import Path

from PIL import Image
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
import glob

from UNet_utils import *

In [ ]:
%load_ext watermark

%watermark
%watermark --iversions

## 2) Configure Inputs and Load U-Net Model

In [ ]:
MODEL_PATH = Path(r"../models/tumor_stroma_unet.h5")
IMAGE_DIR = Path(r"../data/architecture/images")
MASK_DIR = Path(r"../outputs/architecture/tumor_stroma_masks")

In [ ]:
# papermill passes parameters as strings
MODEL_PATH = Path(MODEL_PATH)
IMAGE_DIR = Path(IMAGE_DIR)
MASK_DIR = Path(MASK_DIR)

MASK_DIR.mkdir(parents=True, exist_ok=True)

unet = load_model(
    MODEL_PATH,
    custom_objects={
        "_weighted_binary_crossentropy": weighted_binary_crossentropy(np.ones(2)),
    },
)
Input_size = unet.input_shape[1:3]
print('Model input size: '+str(Input_size[0])+'x'+str(Input_size[1]))

## 3) Run Predictions

In [ ]:
import time

paths = sorted(IMAGE_DIR.glob('*.tif'))
predictions = []

for path in paths:
    start = time.time()
    mask = predict_as_tiles(str(path), unet)
    save_path = MASK_DIR / path.with_suffix('.png').name
    Image.fromarray(mask).save(save_path, "PNG")
    predictions.append(mask)
    end = time.time()
    print(f"Processed {path.name} in {end - start:.2f} seconds")